# Deep Research

### By: Bruno Conterato

## 0. Imports and Config


In [1]:
from agents import (
    Agent,
    Runner,
    function_tool,
    OpenAIChatCompletionsModel,
    AsyncOpenAI,
    ModelSettings,
)
from pydantic import BaseModel
from pydantic.dataclasses import dataclass
import asyncio

In [2]:
@dataclass
class CONFIG:
    model: str = "qwen2.5"
    num_search_terms: int = 3
    debug_mode: bool = False

    query_example: str = "What was the last 5 champions of the NFL Super Bowl?"

## 1. Tools and Definitions


### 1.1. Model to run Ollama

In [3]:
ollama_client = AsyncOpenAI(base_url="http://localhost:11434/v1")

model = OpenAIChatCompletionsModel(model=CONFIG.model, openai_client=ollama_client)

### 1.2. The tool for web search

In [4]:
import httpx


@function_tool
async def search_web(query: str) -> str:
    """Search the web for information about a given query."""

    async with httpx.AsyncClient() as client:
        try:
            response = await client.get(
                "http://localhost:4479/search/text", params={"query": query}
            )
            response.raise_for_status()
            results_arr = [
                f"Title: {r.title}\nBody: {r.body}" for r in response.json()["results"]
            ]
            return "\n\n".join(results_arr)
        except Exception as e:
            return f"Search failed: {str(e)}"

## 2. The Agents Definitions

### 2.1. The search planner Agent

In [5]:
from pydantic import Field


class WebSearchItem(BaseModel):
    reason: str = Field(
        description="Your reasoning for why this search is important to the query."
    )

    query: str = Field(description="The search term to use for the web search.")


class WebSearchPlan(BaseModel):
    searches: list[WebSearchItem] = Field(
        description="A list of web searches to perform to best answer the query."
    )


planner_agent = Agent(
    name="Planner",
    instructions=f"You are a helpful research assistant. Given a query, come up with a set of web searches \
to perform to best answer the query. Output {CONFIG.num_search_terms} terms to query for.",
    model=model,
    output_type=WebSearchPlan,
)

In [6]:
# results = await Runner.run(planner_agent, CONFIG.query_example)

In [7]:
# results.final_output

### 2.2. The search Agent

In [8]:
search_agent = Agent(
    name="Search agent",
    instructions="You are a research assistant. Given a search term, you search the web for that term and \
produce a concise summary of the results. The summary must 2-3 paragraphs and less than 300 \
words. Capture the main points. Write succintly, no need to have complete sentences or good \
grammar. This will be consumed by someone synthesizing a report, so it's vital you capture the \
essence and ignore any fluff. Do not include any additional commentary other than the summary itself.",
    tools=[search_web],
    model=model,
    model_settings=ModelSettings(tool_choice="required"),
)

In [9]:
# results = await Runner.run(
#     search_agent, CONFIG.query_example
# )

In [10]:
# results.final_output

### 2.3. Writter Agent

In [11]:
class ReportData(BaseModel):
    short_summary: str = Field(
        description="A short 2-3 sentence summary of the findings."
    )

    markdown_report: str = Field(description="The final report")

    follow_up_questions: list[str] = Field(
        description="Suggested topics to research further"
    )


writer_agent = Agent(
    name="WriterAgent",
    instructions="You are a senior researcher tasked with writing a cohesive report for a research query. "
    "You will be provided with the original query, and some initial research done by a research assistant.\n"
    "You should first come up with an outline for the report that describes the structure and "
    "flow of the report. Then, generate the report and return that as your final output.\n"
    "The final output should be in markdown format, and it should be lengthy and detailed. Aim "
    "for 5-10 pages of content, at least 1000 words.",
    model=model,
    output_type=ReportData,
)

## 3. Agent Runner Functions


In [12]:
async def plan_searches(query: str) -> WebSearchPlan:
    if CONFIG.debug_mode:
        print("Planning searches...")
        print(f"User query: {query}")
    result = await Runner.run(planner_agent, f"Query: {query}")
    if CONFIG.debug_mode:
        print(f"Will perform searches: {result.final_output}")
    return result.final_output


async def process_search_query(item: WebSearchItem) -> str:
    search_result = await Runner.run(search_agent, item.query)
    return search_result.final_output


async def perform_searches(searchPlan: WebSearchPlan) -> str:
    if CONFIG.debug_mode:
        print("\nPerforming Searches...")
        print(f"Search plan: {searchPlan}")
    tasks = [asyncio.create_task(process_search_query(s)) for s in searchPlan.searches]
    results = await asyncio.gather(*tasks)
    response = ""
    for i, result in enumerate(results):
        response += f"Resultado da busca {i+1}:\n{result}"
    return response


async def write_report(info: str) -> ReportData:
    result = await Runner.run(writer_agent, info)
    return result.final_output

In [13]:
async def deep_research(query: str) -> ReportData:
    searchPlan: WebSearchPlan = await plan_searches(query)
    searchResults = await perform_searches(searchPlan)
    report = await write_report(searchResults)
    return report

In [14]:
result = await deep_research(CONFIG.query_example)

In [15]:
from IPython.display import display, Markdown

display(Markdown(result.markdown_report))

# Recent Super Bowl Outcomes: Analysis and Insights
## Introduction
This report delivers an in-depth analysis of recent Super Bowl outcomes, focusing specifically on the games played in 2023 and 2022. It also provides broader insights into past champions and recurring themes within the NFL.

## Methodology
The research draws from various data points including historical game results, key player performances, and public discussions to provide a nuanced understanding of recent Super Bowl events.

## 2023 Super Bowl Analysis
### Date and Location
- **Date:** February 12, 2023
- **Location:** SoFi Stadium, Inglewood, Los Angeles, California

### Teams
- **Cincinnati Bengals** vs. 
- **Kansas City Chiefs**

### Outcome
- Winning Team: Cincinnati Bengals with a score of 23-20.

### Notable Moments
- The game featured several controversial plays that sparked debate among fans and commentators alike.
- Clutch performances by both teams highlighted key moments in the match.

## 2022 Super Bowl Analysis
### Date and Location
- **Date:** February 13, 2022 (Super Bowl LVI)
- **Location:** SoFi Stadium, Inglewood, Los Angeles, California

### Teams
- **Tampa Bay Buccaneers** vs. 
- **Los Angeles Rams**

### Outcome
- Winning Team: Tampa Bay Buccaneers with a score of 1-28.
- This game was particularly notable due to the performance of quarterback Tom Brady and his achievements.
- Key Player: Tom Brady (Tampa Bay Buccaneers) & Aaron Donald (Los Angeles Rams)

## Historical Context and Analysis of Past Champions
### Recent Super Bowl Successes
#### 2019 - 2023: Current Champions
- **Cincinnati Bengals**: First championship in franchise history; won against Kansas City Chiefs.
- **Tampa Bay Buccaneers**: Ninth championship for Tom Brady, including his first after switching teams and retiring.

### Past Super Bowl Winners (Over the Last Decade)
#### Notable Teams and Years
1. **Philadelphia Eagles** - 2017
2. **Atlanta Falcons** - 2016
3. **Denver Broncos** - 2015
4. **Seattle Seahawks** - 2014

### Recurring Themes in the NFL
- **Defensive Wins**: Key performances by defensive players, such as Aaron Donald and Von Miller.
- **Quarterback Performance**: Impact of key quarterbacks like Tom Brady, Patrick Mahomes, and Justin Herbert.
- **Recurring Conflicts**: Historically tough and occasionally controversial games, particularly involving teams with a strong rivalry or recent history to overcome.

## Discussion on Halftime Shows and Public Interest
### Halftime Show Performers in the 2023 Super Bowl
- Anyakids discussed potential performers for the halftime show, emphasizing the importance of captivating pre-game entertainment.
- Potential performers included mainstream artists known for their ability to engage large audiences during special events such as the Super Bowl.

## Conclusion and Future Predictions
### Implications for Fans and Teams
- The outcomes of recent Super Bowls highlight the importance of strong team performances, particularly in defense, and the unexpected nature of championship victories.
- Future predictions suggest continued dominance by established teams like the Bengals and Chiefs while maintaining interest through strong individual player performances and exciting games.

### Recommendations for Future Research
- Continue monitoring public discussions to gauge fan interest in specific players or teams.
- Analyze future game outcomes and their impact on team strategies, coaching decisions, and player morale.